In [1]:
from dotenv import load_dotenv
load_dotenv('.env')

import warnings
warnings.filterwarnings("ignore")

from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

In [2]:
# Loading Local Module
from src import NewsFetcherInput,NewsFetcher
from src.base import BaseAgent
from src.rss_parser import RSSParsingInput,RSSParser
from src.news_filter import NewsFilterInput,NewsFilterOutput,StockNewsFilter
from src.entity_extractor import EntityExtractionInput,EntityExtraction
from src.plsql_uploader import UploadingInput, UploadToPLSQL

In [ ]:
model_id = 'Qwen/Qwen3-4B-Instruct-2507'
tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype="bfloat16",

)
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    temperature=0.3,
    max_new_tokens=512,
    max_length=None,
)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 398/398 [00:00<00:00, 1352.33it/s]
[transformers] Passing `generation_config` together with generation-related arguments=({'temperature', 'max_length', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [ ]:
import pandas as pd
import json
from tqdm import tqdm
from dataclasses import dataclass,field
from abc import ABC, abstractmethod
from typing import Any, Dict, Optional,Union,Literal
from transformers import Pipeline

class StockNewsFilter(BaseAgent):
    def __init__(self,model:Pipeline):
        super().__init__("Stock News Filtering")
        self.model=model

    async def execute(self,input_data:NewsFilterInput) -> NewsFilterOutput:
        input_data.date = pd.to_datetime(input_data.date,format=input_data.date_format)
        dataset = input_data.dataset
        dataset = dataset[dataset['date']>=input_data.date.date()]
        dataset['is_market_news']=None
        dataset['market_confidence']=None
        dataset['sector']=None
        dataset['reason']=None

        for idx,row in  tqdm(dataset.iterrows(),total=len(dataset)):
            prompt = self.__get_prompt(row)
            result = self.model(
                prompt,
            )
            try:
                model_output = json.loads(result[0]['generated_text'][-1]['content'])
                dataset.at[idx, 'is_market_news'] = model_output.get('is_market_news',None)
                dataset.at[idx, 'market_confidence'] = model_output.get('confidence',None)
                dataset.at[idx, 'sector'] = model_output.get('sector',None)
                dataset.at[idx, 'reason'] = model_output.get('reason',None)
            except Exception as e:
                print(result[0]['generated_text'][-1]['content'])
                self.logger.error(f"Failed for {idx} with model output : {model_output} with error {e}")
                
        dataset.dropna(inplace=True)
        dataset = dataset[dataset['is_market_news']==True]

        self.logger.info(f'Total Newses contains market Data : {dataset.shape[0]}')
        return NewsFilterOutput(
            dataset = dataset
            )

    
    def __get_system_prompt(self):
        return """
You are a financial news classifier. Your task is to determine if the following news is related to stock market or not. 

The Stock market news we can seperate in multiple sector : 
    - IT Sector
    - Banking and Financial Sector
    - Fast-Movingg Consumer Goods sector
    - Automobile Sector
    - Healthcare & Pharma Sector
    - Energy & Utilities Sector
    - Metals & Miningg Sector
    - Infrastructure & Construction Sector
    - Telecomunications Sector
    - Real Estate Sector

You can also determin if the article is related to any of the followings : 
- Economic Impact Analysis
- Global Government policies or SEBI Regulations
- NSE/BSE Changes

If the news article falls into the above 3 reports then you can consider it as "Global Sector"


Return JSON only.

{
  "is_market_news": true if the news is stock market news, else false,
  "confidence": 0.0,
  "sector": provide any one of the sectors mentioned above you think is most suitable for this article,
  "reason": reason for deciding the sector,
}
"""
    def __get_prompt(self,row):
        prompt = """
Article:

Title:
{title}

Content:
{content}
"""
        message = [
                {'role':'system', 'content' : self.__get_system_prompt()},
                {'role':'user','content':prompt.format(title=row.title,content=row.content)}
            ]
        return message
    

In [6]:
input_data = NewsFetcherInput(
    news_link = [
        'https://www.etnownews.com/feeds/gns-etn-companies.xml',
        # 'https://www.etnownews.com/feeds/gns-etn-latest.xml',
        # 'https://www.etnownews.com/feeds/gns-etn-news.xml',
        # 'https://www.etnownews.com/feeds/gns-etn-markets.xml',
        # 'https://www.etnownews.com/feeds/gns-etn-companies.xml',
        # 'https://www.etnownews.com/feeds/gns-etn-personal-finance.xml',
        # 'https://www.etnownews.com/feeds/gns-etn-cryptocurrency.xml',
        # 'https://www.etnownews.com/feeds/gns-etn-news-letter.xml',
        # 'https://www.etnownews.com/feeds/gns-etn-technology.xml',
        # 'https://www.etnownews.com/feeds/gns-etn-economy.xml',
        # 'https://www.etnownews.com/feeds/gns-etn-real-estate.xml',
        # 'https://www.etnownews.com/feeds/gns-etn-budget.xml',
        # 'https://www.etnownews.com/feeds/gns-etn-success-stories.xml',
        ],
    scrapper_config={
        'item_tag':'item',
        'id_tag' : 'guid',
        'title_tag': 'title',
        'description_tag':'description',
        'content_tag':'content:encoded',
        'date_tag':'pubDate',
    },
    scrapper_date_format="%a, %d %b %Y %H:%M:%S %z",
    filter_date = '01-06-2025',
    link_for_listed_company = "https://archives.nseindia.com/content/indices/ind_nifty50list.csv",
    listed_companies_company_name_col = 'Company Name',
    sql_uploading_columns= ['id',"title","content","date","is_market_news","market_confidence","company_name","company_type","company_confidance"],
    sql_column_names=['id','news_id','title','content','published_at','is_market_news','market_confidence_score','company_name','company_type','company_confidence_score'],
    news_id_column='news_id',
    table_name='stock_news_dataset',
    batch_size=2,
)

In [ ]:


# entity_extractor = EntityExtraction(pipe)
# psql_uploader = UploadToPLSQL()

In [7]:
scrapper = RSSParser()
scrapper_input = RSSParsingInput(
    link=input_data.news_link,
    conf = input_data.scrapper_config,
    date_format=input_data.scrapper_date_format,
)
scrapper_out = await scrapper.execute(scrapper_input)



2026-06-13 07:03:23,367 - INFO - Total Data Fetched From XML(s) : 50


In [8]:
news_filter = StockNewsFilter(pipe)
news_filter_input = NewsFilterInput(
    dataset=scrapper_out.dataset,
    date=input_data.filter_date,
    date_format='%d-%m-%Y',
)
news_filter_out = await news_filter.execute(news_filter_input)

  0%|          | 0/50 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
100%|██████████| 50/50 [03:59<00:00,  4.79s/it]
2026-06-13 07:07:24,472 - INFO - Total Newses contains market Data : 47


In [9]:
news_filter_out.dataset

,id,title,description,content,date,is_market_news,market_confidence,sector,reason
0,154583320,Elon Musk becomes the world's first trillionai...,Elon Musk became the world's first trillionair...,Elon Musk has become the world's first trillio...,2026-06-12,True,0.98,IT Sector,The article primarily discusses SpaceX's IPO a...
1,154583083,SpaceX's IPO soars 29% above issue price as El...,SpaceX shares were indicated to open nearly 29...,Elon Musk's SpaceX roared onto public markets ...,2026-06-12,True,0.98,IT Sector,The article discusses SpaceX's IPO and its sto...
2,154582015,"TCS and Anthropic Collaboration: 50,000 employ...",TCS becomes Anthropic's global premier partner...,Two names that rarely appeared in the same sen...,2026-06-12,True,0.95,IT Sector,The article discusses a strategic partnership ...
3,154581823,Tata Sons board approves FY26 annual accounts ...,Tata Sons' board approved FY26 annual accounts...,"The Tata Sons board meeting has concluded, wit...",2026-06-12,True,0.95,Banking and Financial Sector,The article discusses Tata Sons' status as an ...
5,154579790,Tata Sons Board Meet Today: N Chandrasekaran's...,Tata Sons Board Meet On June 12: Tata Sons he...,Tata Sons Board Meet On June 12: The board of ...,2026-06-12,True,0.95,Infrastructure & Construction Sector,"The article discusses Tata Sons, the holding c..."
6,154578223,Vedanta Group's four demerged businesses set t...,"Vedanta's four demerged businesses, VAML, VOGL...",Vedanta group's four demerged businesses are e...,2026-06-11,True,0.95,Metals & Mining Sector,The article primarily discusses the demerger o...
7,154547024,"'Proud to be working with Reliance', says Mark...",Meta and RIL are set to collaborate and its CE...,Meta Platforms has signed an agreement with Re...,2026-06-10,True,0.95,IT Sector,The article discusses a strategic partnership ...
8,154535747,Vedanta Demerger: Malco Energy renamed Vedanta...,Vedanta Demerger: The name change is a part of...,Vedanta Demerger: Vedanta Ltd has renamed its ...,2026-06-09,True,0.98,Energy & Utilities Sector,The article discusses the renaming and demerge...
9,154532081,Adaniâs Surguja coal mine turned into green ...,Surguja coal mine restored with 1.6 million tr...,Surguja coal mine restored with 1.6 million tr...,2026-06-09,True,0.95,Energy & Utilities Sector,The article discusses Adani Enterprises' ecolo...
10,154530617,TCS AGM 2026: AI next major growth driver for ...,TCS is working with enterprises on the deploym...,Tata Consultancy Services is actively collabor...,2026-06-09,True,0.98,IT Sector,The article discusses Tata Consultancy Service...


In [37]:
entity_extraction_input = EntityExtractionInput(
    dataset = news_filter_out.dataset,
    listed_companies = input_data.link_for_listed_company,
    listed_companies_company_name_col = input_data.listed_companies_company_name_col,
    batch_size=input_data.batch_size,
)

extracted_out = await entity_extractor.execute(entity_extraction_input)

Processing batches: 100%|██████████| 23/23 [00:42<00:00,  1.85s/it]
2026-06-10 21:08:46,753 - INFO - Final Dataset To Upload : 27
2026-06-10 21:08:46,753 - INFO - Final Dataset To Upload : 27
2026-06-10 21:08:46,753 - INFO - Final Dataset To Upload : 27
2026-06-10 21:08:46,753 - INFO - Final Dataset To Upload : 27


In [40]:
extracted_out.dataset.head(3)

,id,title,description,content,date,is_market_news,market_confidence,usefull_for,company_name,company_type,company_confidance
0,154547024,"'Proud to be working with Reliance', says Mark...",Meta and RIL are set to collaborate and its CE...,Meta Platforms has signed an agreement with Re...,2026-06-10,True,0.95,sector analysis,Reliance Industries Ltd.,Listed Company,0.98
1,154535747,Vedanta Demerger: Malco Energy renamed Vedanta...,Vedanta Demerger: The name change is a part of...,Vedanta Demerger: Vedanta Ltd has renamed its ...,2026-06-09,True,0.98,stock market analysis,Vedanta Oil & Gas Ltd.,Listed Company,0.95
2,154530617,TCS AGM 2026: AI next major growth driver for ...,TCS is working with enterprises on the deploym...,Tata Consultancy Services is actively collabor...,2026-06-09,True,0.98,company analysis,Tata Consultancy Services Ltd.,Listed Company,0.99


In [ ]:
news_scrapper_input = NewsFetcherInput(
    news_link = [
        'https://www.etnownews.com/feeds/gns-etn-companies.xml'
        'https://www.etnownews.com/feeds/gns-etn-latest.xml',
        'https://www.etnownews.com/feeds/gns-etn-news.xml',
        'https://www.etnownews.com/feeds/gns-etn-markets.xml',
        'https://www.etnownews.com/feeds/gns-etn-companies.xml',
        'https://www.etnownews.com/feeds/gns-etn-personal-finance.xml',
        'https://www.etnownews.com/feeds/gns-etn-cryptocurrency.xml',
        'https://www.etnownews.com/feeds/gns-etn-news-letter.xml',
        'https://www.etnownews.com/feeds/gns-etn-technology.xml',
        'https://www.etnownews.com/feeds/gns-etn-economy.xml',
        'https://www.etnownews.com/feeds/gns-etn-real-estate.xml',
        'https://www.etnownews.com/feeds/gns-etn-budget.xml',
        'https://www.etnownews.com/feeds/gns-etn-success-stories.xml',
        ],
    scrapper_config={
        'item_tag':'item',
        'id_tag' : 'guid',
        'title_tag': 'title',
        'description_tag':'description',
        'content_tag':'content:encoded',
        'date_tag':'pubDate',
    },
    scrapper_date_format="%a, %d %b %Y %H:%M:%S %z",
    filter_date = '01-06-2025',
    link_for_listed_company = "https://archives.nseindia.com/content/indices/ind_nifty50list.csv",
    listed_companies_company_name_col = 'Company Name',
    sql_uploading_columns= ['id',"title","content","date","is_market_news","market_confidence","company_name","company_type","company_confidance"],
    sql_column_names=['id','news_id','title','content','published_at','is_market_news','market_confidence_score','company_name','company_type','company_confidence_score'],
    news_id_column='news_id',
    table_name='stock_news_dataset',
)

news_scrapper = NewsFetcher(model=pipe,parser='rss')

await news_scrapper.execute(news_scrapper_input)

2026-06-08 01:24:47,345 - INFO - Initiated!
2026-06-08 01:24:47,345 - INFO - Scrapping Newses..
2026-06-08 01:24:48,767 - INFO - Filtering Newses for Stock Market News.
0it [00:00, ?it/s][transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
5it [00:22,  4.50s/it]